# Build a SQL agent


## Overview
In this tutorial, you will learn how to build an agent that can answer questions about a SQL database using LangChain agents.
At a high level, the agent will:

1 Fetch the available tables and schemas from the database

2 Decide which tables are relevant to the question

3 Fetch the schemas for the relevant tables

4 Generate a query based on the question and information from the schemas

5 Double-check the query for common mistakes using an LLM

6 Execute the query and return the results

7 Correct mistakes surfaced by the database engine until the query is successful

8 Formulate a response based on the results

In [1]:
import sys
import os
import openai

# Use current working directory and go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you can import your config
from config import api_key, tavily_api_key

#openaai.api_key = api_key
os.environ['OPENAI_API_KEY'] = api_key
os.environ['TAVILY_API_KEY'] = tavily_api_key

## 1. Select an LLM
Select a model that supports tool-calling:

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1", api_key=api_key)

## 2. Configure the database

You will be creating a SQLite database for this tutorial. SQLite is a lightweight database that is easy to set up and use. We will be loading the chinook database, which is a sample database that represents a digital media store.
For convenience, we have hosted the database (Chinook.db) on a public GCS bucket.

In [3]:
import requests
import pathlib

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
local_path = pathlib.Path("Chinook.db")

In [4]:
if local_path.exists():
    print(f"{local_path} already exists, skipping download.")
else:
    response = requests.get(url)
    if response.status_code == 200:
        local_path.write_bytes(response.content)
        print(f"File downloaded and saved as {local_path}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")

Chinook.db already exists, skipping download.


We will use a handy SQL database wrapper available in the langchain_community package to interact with the database. The wrapper provides a simple interface to execute SQL queries and fetch results:

In [5]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print(f'Sample output: {db.run("SELECT * FROM Artist LIMIT 5;")}')

Dialect: sqlite
Available tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Sample output: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


### 3. Add tools for database interactions
Use the SQLDatabase wrapper available in the langchain_community package to interact with the database. The wrapper provides a simple interface to execute SQL queries and fetch results:

In [6]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



## 4. Use create_agent

Use `create_agent` to build a **ReAct agent** with minimal code. The agent will interpret the request and generate a SQL command, which the tools will execute. If the command has an error, the error message is returned to the model. The model can then examine the original request and the new error message and generate a new command. This can continue until the LLM generates the command successfully or reaches an end count. This pattern of providing a model with feedback - error messages in this case - is very powerful.
Initialize the agent with a descriptive system prompt to customize its behavior:

In [7]:
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect=db.dialect,
    top_k=5,
)

In [14]:
from langchain.agents import create_react_agent
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(system_prompt)

agent = create_react_agent(llm=model,tools=tools, prompt=prompt)

ValueError: Prompt missing required variables: {'agent_scratchpad', 'tools', 'tool_names'}

In [26]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode, tools_condition

# Create the state to capture the messages
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [28]:
llm = init_chat_model("openai:gpt-4.1", api_key=api_key)
llm_with_tools = llm.bind_tools(tools)

def llm_node(State):
    prompt = SystemMessage(content=system_prompt)
    
    
    

In [23]:
builder = StateGraph(State)


In [30]:
prompt = SystemMessage(content=system_prompt)
    

In [31]:
prompt

SystemMessage(content='\nYou are an agent designed to interact with a SQL database.\nGiven an input question, create a syntactically correct sqlite query to run,\nthen look at the results of the query and return the answer. Unless the user\nspecifies a specific number of examples they wish to obtain, always limit your\nquery to at most 5 results.\n\nYou can order the results by a relevant column to return the most interesting\nexamples in the database. Never query for all the columns from a specific table,\nonly ask for the relevant columns given the question.\n\nYou MUST double check your query before executing it. If you get an error while\nexecuting a query, rewrite the query and try again.\n\nDO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the\ndatabase.\n\nTo start you should ALWAYS look at the tables in the database to see what you\ncan query. Do NOT skip this step.\n\nThen you should query the schema of the most relevant tables.\n', additional_kwargs={}, re